In [ ]:
import scipy.interpolate
import teqp
import numpy as np
import pandas
import matplotlib.pyplot as plt
teqp.__version__

# Critical curves & points


## Pure Fluids

Solving for the critical point involves finding the temperature and density that make
$$
\left(\frac{\partial p}{\partial \rho}\right)_T = \left(\frac{\partial^2 p}{\partial \rho^2}\right)_T = 0
$$
by 2D non-linear rootfinding. Newton steps are taken, and the analytic Jacobian is used (thanks to the ability to do derivatives with automatic differentiation).  This is all handily wrapped up in the
``solve_pure_critical`` method which requires the user to provide guess values for temperature and density

In [ ]:
# Values taken from http://dx.doi.org/10.6028/jres.121.011
modelPR = teqp.canonical_PR([190.564], [4599200], [0.011])

# Solve for the critical point from a point close to the critical point
T0 = 192.0
# Critical compressibility factor of P-R is 0.307401308698.. (see https://doi.org/10.1021/acs.iecr.1c00847)
rhoc = (4599200/(8.31446261815324*190.564))/0.3074
rho0 = rhoc*1.2345 # Perturb to make sure we are doing something in the solver
modelPR.solve_pure_critical(T0, rho0)

If you have a mixture, but want to obtain the critical point of a pure fluid of this mixture, you can specify the index of the component in the mixture, as well as the number of componnts in the mixture with something like:

``
model.solve_pure_critical(T0, rho0, {"alternative_pure_index": 1, "alternative_length": 2})
``
so here, for the second fluid, with 0-based index of 1, in a two-component mixture

## Mixtures

A pure fluid has a single vapor-liquid critical point, but mixtures are different:

* They may have multiple (or zero!) critical points for a given mixture composition
* The critical curves may not emanate from the pure fluid endpoints

When it comes to critical points, intuition from pure fluids is not helpful, or sometimes even counter-productive. 

teqp has methods for working with the critical loci of binary mixtures (only binary mixtures, for now) and especially, methods for tracing the critical curves emanating from the pure fluid endpoints.

The tracing method in teqp is based explicitly on the isochoric thermodynamics formalism introduced by Ulrich Deiters and Sergio Quinones-Cisneros.  It uses the Helmholtz energy density as the fundamental potential and all other properties are derived from it.  For critical curves it is based upon the integration of sets of ordinary differential equations; the differential equations are in the form of derivatives of the molar concentrations of each component in the mixture with respect to an integration variable.  The set of ODE is then integrated.

Here is an example of the construction of the critical curves emanating from the pure fluid endpoints for the mixture nitrogen + ethane.

In [ ]:
import timeit
import numpy as np
import matplotlib.pyplot as plt 
import pandas
import teqp

def get_critical_curve(ipure):
    """ Return curve as pandas DataFrame """
    names = ['Nitrogen', 'Ethane']
    model = teqp.build_multifluid_model(names, teqp.get_datapath())
    T0 = model.get_Tcvec()[ipure]
    rho0 = np.array([1.0/model.get_vcvec()[ipure]]*2)
    rho0[1-ipure] = 0
    o = teqp.TCABOptions() 
    o.init_dt = 1.0 # step in the arclength tracing parameter
    o.rel_err = 1e-8
    o.abs_err = 1e-5
    o.integration_order = 5
    o.calc_stability = True
    o.polish = True
    curveJSON = model.trace_critical_arclength_binary(T0, rho0, '', o)
    df = pandas.DataFrame(curveJSON)
    rhotot = df['rho0 / mol/m^3']+df['rho1 / mol/m^3']
    df['z0 / mole frac.'] = df['rho0 / mol/m^3']/rhotot
    return df

fig, ax = plt.subplots(1,1,figsize=(7, 6))
tic = timeit.default_timer()
for ipure in [1,0]:
    df = get_critical_curve(ipure)
    first_unstable = np.argmax(~df['locally stable'])
    df = df.iloc[0:(first_unstable if first_unstable else len(df))]
    line, = plt.plot(df['T / K'], df['p / Pa']/1e6, '.')
    
    # And interpolate to smooth out the curve using the arclength
    # parameter (which must be monotonically increasing) as 
    # the interpolation variable
    tinterp = np.linspace(df['t'].min(), df['t'].max(), 10000)
    Tinterp = scipy.interpolate.interp1d(df['t'], df['T / K'], kind='cubic')(tinterp)
    pinterp = scipy.interpolate.interp1d(df['t'], df['p / Pa'], kind='cubic')(tinterp)
    plt.plot(Tinterp, pinterp/1e6, color=line.get_color())
    
    plt.plot(df['T / K'].iloc[0], df['p / Pa'].iloc[0]/1e6, 'd', 
        color=line.get_color())

elap = timeit.default_timer()-tic
plt.gca().set(xlabel='$T$ / K', ylabel='$p$ / MPa',
    xlim=(100, 350), ylim=(1, 1e3))
plt.yscale('log')
plt.tight_layout(pad=0.2)
plt.gcf().text(0,0,f'time: {elap:0.1f} s', ha='left', va='bottom', fontsize=7)
plt.gcf().text(1,0,f'teqp: {teqp.__version__}', ha='right', va='bottom', fontsize=7);

And now for something a bit more interesting: ethane + alkane critical curves

In [ ]:
import timeit
import numpy as np
import matplotlib.pyplot as plt 
import pandas
import teqp

def get_critical_curve(names, ipure):
    """ Return curve as pandas DataFrame """
    model = teqp.build_multifluid_model(names, teqp.get_datapath())
    T0 = model.get_Tcvec()[ipure]
    rho0 = np.array([1.0/model.get_vcvec()[ipure]]*2)
    rho0[1-ipure] = 0
    o = teqp.TCABOptions() 
#     print(dir(o))
    o.init_dt = 1.0 # step in the parameter
    o.rel_err = 1e-6 # relative error on the step
    o.abs_err = 1e-6 # absolute error on the step
    o.max_dt = 100 # cap the size of the allowed step
    o.calc_stability = True
    o.polish = True
    curveJSON = model.trace_critical_arclength_binary(T0, rho0, '', o)
    df = pandas.DataFrame(curveJSON)
    rhotot = df['rho0 / mol/m^3']+df['rho1 / mol/m^3']
    df['z0 / mole frac.'] = df['rho0 / mol/m^3']/rhotot
    return df

fig, ax = plt.subplots(1,1,figsize=(7, 6))
tic = timeit.default_timer()
name0 = 'ETHANE'
for othername in ['METHANE','PROPANE','BUTANE','PENTANE','HEXANE']:
    for ipure in [1]:
        df = get_critical_curve([name0, othername], ipure)
        line, = plt.plot(df['T / K'], df['p / Pa']/1e6, '-')
        plt.plot(df['T / K'].iloc[0], df['p / Pa'].iloc[0]/1e6, 'd', 
            color=line.get_color())

elap = timeit.default_timer()-tic
plt.gca().set(xlabel='$T$ / K', ylabel='$p$ / MPa')#,xlim=(100, 350), ylim=(1, 1e3))
plt.tight_layout(pad=0.2)
plt.gcf().text(0,0,f'time: {elap:0.1f} s', ha='left', va='bottom', fontsize=7)
plt.gcf().text(1,0,f'teqp: {teqp.__version__}', ha='right', va='bottom', fontsize=7);

## Pure fluid EOS with nonanalytic terms

For the highest accuracy EOS for normal water and carbon dioxide, there are non-analytic terms that prevent the initialization of the critical tracing at the pure fluid critical point. Instead, one can start close to, but not *AT*, the pure fluid endpoint. After deciding on that starting composition, one solves for the critical point and then traces away from it.

You might need to either do tracing in two parts, one with init_c=+1 and then init_c=-1, or one tracing might be good enough.

Here is an example:

In [ ]:
def get_critical_curve_composition(names, T0, rhovec0, init_c=-1):
    """ Trace the critical curve from a fixed point along it """
    o = teqp.TCABOptions() 
#     print(dir(o))
    o.init_dt = 1.0 # step in the parameter
    o.rel_err = 1e-6 # relative error on the step
    o.abs_err = 1e-6 # absolute error on the step
    o.max_dt = 100 # cap the size of the allowed step
    o.calc_stability = True
    o.polish = True
    o.init_c = init_c # You might need to swap the initial tracing direction by making this +1.0
    curveJSON = model.trace_critical_arclength_binary(T0, rhovec0, '', o)
    df = pandas.DataFrame(curveJSON)
    rhotot = df['rho0 / mol/m^3']+df['rho1 / mol/m^3']
    df['z0 / mole frac.'] = df['rho0 / mol/m^3']/rhotot
    return df

# Tracing with multi-fluid from an endpoint with non-analytic terms
model = teqp.build_multifluid_model(["Water", "Methane"], teqp.get_datapath())

x0 = 1-1e-6 # ever so slightly away from the pure fluid
molefrac = np.array([x0, 1-x0])

# Solve for the actual critical point at this mole fraction with scipy
y0 = [model.get_Tcvec()[0], 1/model.get_vcvec()[0]]
residual = lambda y: model.get_criticality_conditions(y[0], y[1]*molefrac)
res = scipy.optimize.fsolve(residual, y0)
T = res[0]
rho0 = res[1]
rhovec0 = rho0*molefrac

# Now trace from this point
curve = get_critical_curve_composition(model, T0=T, rhovec0=rhovec0)
plt.plot(curve['T / K'], curve['p / Pa']/1e6, label='multifluid')

# With GERG-2008, things are much more straightforward...
model = teqp.make_model({'kind': 'GERG2008resid', 'model': {'names': ['water','methane']}})

def get_critical_curve_simple(model, ipure, T0, rho0):
    """ Trace from a pure fluid... """
    rhovec0 = np.array([0, 0])
    rhovec0[ipure] = rho0
    o = teqp.TCABOptions()
    o.init_dt = 1.0 # step in the arclength tracing parameter
    o.rel_err = 1e-8
    o.abs_err = 1e-5
    o.integration_order = 5
    o.calc_stability = True
    o.polish = True
    curveJSON = model.trace_critical_arclength_binary(T0, rhovec0, '', o)
    df = pandas.DataFrame(curveJSON)
    rhotot = df['rho0 / mol/m^3']+df['rho1 / mol/m^3']
    df['z0 / mole frac.'] = df['rho0 / mol/m^3']/rhotot
    return df

for ifluid in [0]:
    Tci = model.get_Tcvec()[ifluid]
    vci = model.get_vcvec()[ifluid]
    df = get_critical_curve_simple(model, ipure=ifluid, T0=Tci, rho0 = 1.0/vci)
    plt.plot(df['T / K'], df['p / Pa']/1e6, label='GERG-2008')

plt.gca().set(xlabel='$T$ / K', ylabel='$p$ / MPa')
plt.yscale('log')
plt.xlim(600, 950)
plt.ylim(20, 300)
plt.legend(loc='best');

## Critical points

If you have a relatively simple critical curve for a binary mixture and you would like to obtain the critical point at a given composition, you can obtain it reliably in a multi-step process:

1. Trace the binary curve 
2. Interpolate to find the approximate value for critical temperature and density
3. Use Newton's method to solve the criticality conditions from estimated critical point
4. Calculate whatever else you want

In [ ]:
df = get_critical_curve([name0, othername], ipure)
model = teqp.build_multifluid_model([name0, othername], teqp.get_datapath())
line, = plt.plot(df['T / K'], df['p / Pa']/1e6, '-')
rhotot = df['rho0 / mol/m^3']+df['rho1 / mol/m^3']

# Build cubic interpolators between the variables
tinterp = scipy.interpolate.interp1d(df['z0 / mole frac.'], df['t'], kind='cubic') # interpolator from mole fraction to tracing parameter
Tinterp = scipy.interpolate.interp1d(df['t'], df['T / K'], kind='cubic')
rhointerp = scipy.interpolate.interp1d(df['t'], rhotot, kind='cubic')

# Iterate over composition
for x0 in np.arange(0.1, 1, 0.05):

    molefracs = np.array([x0, 1-x0])
        
    # Initial guess for critical point from interpolation
    t = tinterp(x0)
    T0 = Tinterp(t)
    rho0 = rhointerp(t)
    p0 = rho0*model.get_R(molefracs)*T0*(1+model.get_Ar01(T0, rho0, molefracs))
    plt.plot(T0, p0/1e6, 'o')
    
    # Newton iteration for the correct critical point
    def resid(x):
        return model.get_criticality_conditions(T=x[0], rhovec=x[1]*molefracs)
    soln = scipy.optimize.fsolve(resid, x0=[T0, rho0])
    T, rho = soln
    p = rho*model.get_R(molefracs)*T*(1+model.get_Ar01(T, rho, molefracs))
    plt.plot(T, p/1e6, '+')
#     print(x0, T0, rho0, p0, resid([T0, rho0]), T, rho, resid([T, rho]))

elap = timeit.default_timer()-tic
plt.gca().set(xlabel='$T$ / K', ylabel='$p$ / MPa')
plt.tight_layout(pad=0.2)

## Tangent of the critical curve

`trace_critical_arclength_binary` follows the critical curve of a binary mixture in the variables $(T, \rho_0, \rho_1)$ and needs the tangent ${\rm d}\vec\rho/{\rm d}T$ of the curve at every step. `get_drhovec_dT_crit` returns that tangent as the null vector of the Jacobian of the criticality conditions. The Jacobian is `CriticalTracing::get_criticality_conditions_Jacobian` in the C++ header `teqp/algorithms/critical_tracing.hpp`. This section derives it.

With $\Psi = \Psi^0 + \Psi^{\rm r}$ the Helmholtz energy density, $\mathbf H$ its Hessian in the molar concentrations, $\lambda_0 < \lambda_1$ the eigenvalues of $\mathbf H$ and $\vec v_0, \vec v_1$ its eigenvectors, the conditions of Deiters and Bell (https://dx.doi.org/10.1021/acs.iecr.0c03667) are

$$C_1 = \lambda_0 = 0, \qquad C_2 = \frac{\partial^3 \Psi}{\partial \sigma_1^3} = 0,$$

where $\sigma_1$ and $\sigma_2$ are the coordinates along $\vec v_0$ and $\vec v_1$. Below, a subscript 0 or 1 is a derivative along $\vec v_0$ or $\vec v_1$ with the direction held fixed, and a subscript $T$ is a temperature derivative.

### Derivatives of the conditions

$\lambda_0 = \vec v_0^{\rm T}\mathbf H\vec v_0$ is stationary with respect to $\vec v_0$, so for $\xi \in \{\sigma_1, \sigma_2, T\}$

$$\frac{\partial C_1}{\partial \xi} = \vec v_0^{\rm T}\frac{\partial \mathbf H}{\partial \xi}\vec v_0 = \Psi_{000},\ \Psi_{001},\ \Psi_{00T}.$$

$C_2$ is a cubic form in $\vec v_0$, and $\vec v_0$ turns with the state. First-order perturbation of the eigenvector gives

$$\frac{\partial \vec v_0}{\partial \xi} = \frac{\vec v_1^{\rm T}(\partial \mathbf H/\partial \xi)\vec v_0}{\lambda_0 - \lambda_1}\,\vec v_1,$$

and $\partial C_2/\partial \vec v_0 \cdot \vec v_1 = 3\Psi_{001}$, so

$$\begin{aligned}
\frac{\partial C_2}{\partial \sigma_1} &= \Psi_{0000} + \frac{3\Psi_{001}^2}{\lambda_0 - \lambda_1} \\
\frac{\partial C_2}{\partial \sigma_2} &= \Psi_{0001} + \frac{3\Psi_{001}\Psi_{011}}{\lambda_0 - \lambda_1} \\
\frac{\partial C_2}{\partial T} &= \Psi_{000T} + \frac{3\Psi_{001}\Psi_{01T}}{\lambda_0 - \lambda_1}.
\end{aligned}$$

The derivatives with respect to the concentrations follow from $\partial C/\partial \vec\rho = (\partial C/\partial\sigma_1)\vec v_0^{\rm T} + (\partial C/\partial\sigma_2)\vec v_1^{\rm T}$. The tangent of the critical curve is the null vector of the $2\times 3$ Jacobian $\partial C/\partial(T, \rho_0, \rho_1)$.

In Python the tangent is `get_drhovec_dT_crit`. The cell below evaluates it at a critical point of nitrogen + ethane and compares it with the difference of two neighbouring critical points.

In [ ]:
model = teqp.build_multifluid_model(['Nitrogen', 'Ethane'], teqp.get_datapath())

def critical_point(x0, T, rho):
    """ Solve the criticality conditions for T and the molar density at the mole fraction x0 of nitrogen """
    molefracs = np.array([x0, 1-x0])
    T, rho = scipy.optimize.fsolve(lambda x: model.get_criticality_conditions(x[0], x[1]*molefracs), x0=[T, rho], xtol=1e-13)
    return T, rho*molefracs

# A critical point close to pure ethane, starting from the critical point of ethane
x0, dx = 0.05, 1e-4
T, rhovec = critical_point(x0, model.get_Tcvec()[1], 1/model.get_vcvec()[1])
print('T / K:', T, 'rhovec / mol/m^3:', rhovec)
print('get_drhovec_dT_crit:                 ', model.get_drhovec_dT_crit(T, rhovec))

# The same derivative from the critical points at x0 - dx and x0 + dx
Tm, rhovecm = critical_point(x0-dx, T, rhovec.sum())
Tp, rhovecp = critical_point(x0+dx, T, rhovec.sum())
print('difference of solved critical points:', (rhovecp-rhovecm)/(Tp-Tm))

### A gas constant that depends on composition

The ideal-gas part of $\mathbf H$ and of $C_2$ is built with the value of $R(\vec x)$ at the state. With a constant $R$ the formulas above are complete. Where $R$ depends on composition (the mole-fraction-weighted $R$ of the multifluid model, when the pure-fluid equations use different gas constants), the conditions depend on the concentrations through $R$ as well, and $\partial C/\partial\vec\rho$ gains the term $(\partial C/\partial R)(\nabla_{\vec\rho}R)^{\rm T}$ with

$$\frac{\partial C_1}{\partial R} = T\sum_i \frac{v_{0,i}^2}{\rho_i}, \qquad
\frac{\partial C_2}{\partial R} = -T\sum_i \frac{v_{0,i}^3}{\rho_i^2} + \frac{3\Psi_{001}}{\lambda_0 - \lambda_1}\,T\sum_i \frac{v_{0,i}v_{1,i}}{\rho_i}.$$

For methane + ethane (8.31451 and 8.314472 J/(mol K)) this term changes the tangent ${\rm d}\vec\rho/{\rm d}T$ by 2e-6 to 1.5e-5 (relative) at $x_{\rm CH_4}$ = 0.3 and 0.7.

### Mixed derivatives from dual numbers

Automatic differentiation gives every mixed derivative directly. In this subsection every $\Psi$ subscript is the residual part.

The C++ model method `get_Psir_sigma_derivs` returns the derivatives of order up to 4 of $s \mapsto \Psi^{\rm r}(T, \vec\rho + s\,\vec v_0)$, which gives $\Psi_{000}$ and $\Psi_{0000}$. The C++ model method `get_Psir_mixed_dir_derivs` evaluates $\Psi^{\rm r}$ once with nested dual numbers, one level per direction in the joint space of temperature and molar concentrations, and returns the mixed derivatives along those directions. A direction given on several levels gives the higher derivative along it. With the levels along

* $(\vec v_0, \vec v_0, \vec v_0, T)$: $\Psi_{000T}$ and $\Psi_{00T}$,
* $(\vec v_0, \vec v_0, \vec v_0, \vec v_1)$: $\Psi_{0001}$ and $\Psi_{001}$,
* $(\vec v_0, \vec v_1, \vec v_1)$: $\Psi_{011}$,
* $(\vec v_0, \vec v_1, T)$: $\Psi_{01T}$,

one Jacobian takes one derivative of order 4, two evaluations with dual numbers of order 4, two of order 3, and the Hessian of $\Psi^{\rm r}$ for the eigenproblem.

Each mixed derivative is the coefficient of its own product of dual units.

The ideal-gas part is added separately. It is $\Psi^0 = RT\sum_i \rho_i(\ln\rho_i - 1)$ up to terms that are linear in the molar concentrations, and those drop out of every derivative used here, which are all of order two or higher in the concentrations. Its derivative tensors are diagonal,

$$\Psi^0_{iii} = -\frac{RT}{\rho_i^2}, \quad \Psi^0_{iiii} = \frac{2RT}{\rho_i^3}, \quad
\Psi^0_{iiT} = \frac{R}{\rho_i}, \quad \Psi^0_{iiiT} = -\frac{R}{\rho_i^2},$$

and they are contracted with $\vec v_0$ and $\vec v_1$ term by term, for example $\Psi^0_{001} = -RT\sum_i v_{0,i}^2 v_{1,i}/\rho_i^2$. The ideal part is not taken with dual numbers along the eigenvectors: at a minor mole fraction $x$ its derivatives of order 3 along the minor component are about $1/x^2$ times the others, and they would be subtracted from each other.

### The limit at a pure fluid

When one molar concentration is exactly zero, the criticality conditions are evaluated from the reduced eigenproblem of the present component, as described by Deiters and Bell: the eigenvalue $\lambda_1 \sim RT/\rho_q$ of the absent component $q$ diverges and is left out. The derivatives of the conditions with respect to $\rho_q$ are then not available from the formulas above, which contain $\lambda_1$ and $\vec v_1$. They have finite limits, and `get_criticality_conditions_Jacobian` uses these limits at $\rho_q = 0$. At a small $\rho_q$ that is not zero it uses the formulas above. The component of $\vec v_0$ along $q$ is then of the order of $\rho_q$ and enters divided by $\rho_q$, so the eigenvectors of the $2\times 2$ Hessian are taken from one Jacobi rotation in closed form, which gives that component to relative accuracy, and the ideal-gas terms are formed from the ratios $v_{0,i}/\rho_i$ and $v_{1,i}/\rho_i$. With $\vec v_0$ oriented as at $\rho_q = 0$ (the second condition is odd in $\vec v_0$), the Jacobian is continuous from $\rho_q = 0$ on.

Expanding in $\rho_q$, with $p$ the present component and $\mathbf H^{\rm r}$ the Hessian of $\Psi^{\rm r}$,

$$\lambda_0 = H_{pp} - \frac{\rho_q (H^{\rm r}_{pq})^2}{RT} + O(\rho_q^2), \qquad
\vec v_0 = \vec e_p - \frac{\rho_q H^{\rm r}_{pq}}{RT}\vec e_q + O(\rho_q^2),$$

so the conditions and the Jacobian have the limits below, in which the subscripts are plain component derivatives of $\Psi^{\rm r}$:

$$\begin{aligned}
C_1 &= \frac{RT}{\rho_p} + \Psi^{\rm r}_{pp}, \qquad C_2 = -\frac{RT}{\rho_p^2} + \Psi^{\rm r}_{ppp} \\
\frac{\partial C_1}{\partial T} &= \frac{R}{\rho_p} + \Psi^{\rm r}_{ppT}, \qquad
\frac{\partial C_1}{\partial \rho_p} = C_2, \qquad
\frac{\partial C_1}{\partial \rho_q} = \Psi^{\rm r}_{ppq} - \frac{(\Psi^{\rm r}_{pq})^2}{RT} \\
\frac{\partial C_2}{\partial T} &= -\frac{R}{\rho_p^2} + \Psi^{\rm r}_{pppT}, \qquad
\frac{\partial C_2}{\partial \rho_p} = \frac{2RT}{\rho_p^3} + \Psi^{\rm r}_{pppp} \\
\frac{\partial C_2}{\partial \rho_q} &= \Psi^{\rm r}_{pppq} - \frac{3\Psi^{\rm r}_{ppq}\Psi^{\rm r}_{pq}}{RT} + \frac{(\Psi^{\rm r}_{pq})^3}{(RT)^2}.
\end{aligned}$$

The first line is the pair of critical conditions of the pure fluid. The derivatives with respect to $\rho_q$ give the direction in which the critical curve leaves the pure fluid. The term for a composition-dependent gas constant has the limit $\partial C/\partial R = [T/\rho_p,\ -T/\rho_p^2]$.

A mixture that contains a fluid whose equation of state has nonanalytic terms (carbon dioxide, water; see the section on these equations above) is an exception at the pure-fluid end of that fluid. There $\delta = \tau = 1$, the Jacobian contains mixed temperature and density derivatives of the nonanalytic terms, which do not exist at that point, and the expressions above are not the limit of the mixture Jacobian. For carbon dioxide + ethane the tangent at pure carbon dioxide depends on the last digits of the state. The unit tangents (in $T/T_{\rm c}$ and $\rho/\rho_{\rm c}$ of the pure fluid) at pure carbon dioxide and at $x_{\rm C_2H_6} = 10^{-12}$ differ by 0.48 at $T_{\rm c}$ and $1/v_{\rm c}$ of the fluid file, and by $3.6\times10^{-5}$ at the solved critical point, whose density differs from $1/v_{\rm c}$ by $8\times10^{-8}$ (relative). Start a trace from that end next to the pure fluid, as in the example above. At the other end the fluid with the nonanalytic terms is absent, its terms enter the derivatives with respect to $\rho_q$ through their value and their density derivatives only, and the expressions above are the limit: the unit tangents at pure ethane and at $x_{\rm CO_2} = 10^{-12}$ differ by $7\times10^{-11}$.